In [0]:
from pyspark.sql.functions import col, upper, trim, initcap, length, abs, when, lit, row_number, desc
from pyspark.sql.window import Window

# 1. Read from Bronze
df = spark.read.table("hdfc_ergo.hdfc_bronze.dim_members")

# 2. Safe Type Casting (Strings to Decimals)
df = df.withColumn("sum_insured", col("sum_insured").cast("DECIMAL(15,2)")) \
       .withColumn("risk_score", col("risk_score").cast("DECIMAL(5,2)"))

# 3. Text Standardization (UPPER/TRIM for IDs and categories)
df = df.withColumn("member_id", upper(trim(col("member_id")))) \
       .withColumn("policy_number", upper(trim(col("policy_number")))) \
       .withColumn("state", upper(trim(col("state")))) \
       .withColumn("city", upper(trim(col("city")))) \
       .withColumn("employment_status", upper(trim(col("employment_status"))))

# Proper case for Names (Changed from 'name' to 'full_name' to match your schema)
df = df.withColumn("full_name", initcap(trim(col("full_name"))))

# 4. Validate Pincode and Aadhar (Changed from 'aadhar' to 'aadhar_number' to match your schema)
df = df.withColumn("pincode", when((length(trim(col("pincode"))) == 6), trim(col("pincode"))).otherwise(lit(None))) \
       .withColumn("aadhar_number", when((length(trim(col("aadhar_number"))) == 12), trim(col("aadhar_number"))).otherwise(lit(None)))

# 5. Business Logic: Fix negative financials
df = df.withColumn("sum_insured", when(col("sum_insured") < 0, abs(col("sum_insured"))).otherwise(col("sum_insured")))

# 6. Deduplication (Changed window ordering to use 'enrollment_date' instead of missing '_ingested_at')
windowSpec = Window.partitionBy("member_id").orderBy(desc("enrollment_date"))
df = df.withColumn("_dq_is_deduped", row_number().over(windowSpec))

# Filter to keep only the first record (latest), drop the helper column
df_deduped = df.filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped")

# 7. Write to Silver Schema
df_deduped.write.mode("overwrite").saveAsTable("hdfc_ergo.hdfc_silver.dim_members")

print("✅ Silver dim_members created successfully!")
display(spark.table("hdfc_ergo.hdfc_silver.dim_members").limit(10))